In [2]:
# import libraries 
import time
import re
import datetime
import csv
import logging
import sys
import os

from selenium import webdriver
from bs4 import BeautifulSoup

# cần 2 cái dưới để điều hướng con trỏ chuột sang trang kế
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import logging # dùng để ghi tiến trình & lỗi khi crawl data

# Import 3 vũ khí mới để vượt qua bước Login của web
import os # để đọc thông tin trong file `.env`
import undetected_chromedriver as uc
from dotenv import load_dotenv

In [3]:
# --- Thiết lập logging ---
logging.basicConfig(
    filename="crawl_jobs_itviec.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding='utf-8'
)

# 1. LOGIN vào web

In [4]:
# Kích hoạt đọc file ẩn .env
load_dotenv()

True

In [5]:
# Lấy Email và Pass từ file .env ra (Người khác nhìn code sẽ không thấy pass)
MY_EMAIL = os.getenv("ITVIEC_EMAIL")
MY_PASSWORD = os.getenv("ITVIEC_PASS")
MY_EMAIL

'Huyen.TTM206288@sis.hust.edu.vn'

In [6]:
KEYWORDS = ["Data Analyst"] # Rút gọn để test
BASE_URL_TEMPLATE = "https://itviec.com/it-jobs/{keyword}?page={page}"
MAX_PAGES = 3
OUTPUT_CSV = "itviec_jobs.csv"

# ── 3. LOGGING (Giữ nguyên của bạn) ──────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

In [7]:
# THAY ĐỔI 1: Dùng undetected_chromedriver thay vì webdriver.Chrome()
options = uc.ChromeOptions()


# Thêm cấu hình vô hiệu hóa chức năng hỏi lưu mật khẩu
prefs = {
    "credentials_enable_service": False,
    "profile.password_manager_enabled": False
}
options.add_experimental_option("prefs", prefs)


# THÊM version_main=140 VÀO ĐÂY vì nó mặc định chạy ver Chrome 147
driver = uc.Chrome(options=options, version_main=140) 


# Thay vì maximize_window(), dùng set_window_size và set_window_position
# Đặt kích thước cho trình duyệt (chiều rộng x chiều cao)
driver.set_window_size(900, 1080) # Bạn có thể thay đổi số theo độ phân giải màn hình
# (Tùy chọn) Đặt vị trí xuất hiện của cửa sổ (x, y) trên màn hình (0,0 là góc trên cùng bên trái)
driver.set_window_position(900, 0) # Có thể đẩy nó sang lề phải của màn hình

{'height': 882, 'width': 903, 'x': 900, 'y': 0}

In [8]:
# =================================================================
# THAY ĐỔI 2: THỰC HIỆN ĐĂNG NHẬP TRƯỚC KHI CÀO
# =================================================================
print("Đang tiến hành đăng nhập ITviec")
driver.get("https://itviec.com/sign_in")

wait = WebDriverWait(driver, 15)

Đang tiến hành đăng nhập ITviec


In [9]:
# Điền Email
email_input = wait.until(EC.presence_of_element_located((By.ID, "user_email")))
email_input.send_keys(MY_EMAIL)

# Điền Password
pass_input = driver.find_element(By.ID, "user_password")
pass_input.send_keys(MY_PASSWORD)

In [10]:
# Bấm nút Sign In with Email
submit_btn = driver.find_element(
    By.XPATH, 
    "//button[@type='submit' and .//span[text()='Sign In with Email']]"
    )
submit_btn.click()

print("Đăng nhập thành công! Chờ 5s để web ổn định...")
time.sleep(5)

Đăng nhập thành công! Chờ 5s để web ổn định...


In [11]:
# ==========================================
# ĐOẠN CODE XỬ LÝ TẮT POPUP "TẠO CV"
# ==========================================
try:
    # Tìm phần tử có chứa chữ "Remind me later"
    # Dùng //* để bắt mọi loại thẻ (button, a, span...) có chứa chữ này
    close_popup_btn = driver.find_element(By.XPATH, "//*[contains(text(), 'Remind me later')]")
    
    # Dùng Javascript để click cho chắc ăn, xuyên qua mọi lớp che khuất
    driver.execute_script("arguments[0].click();", close_popup_btn)
    
    logging.info("Đã đóng popup yêu cầu tạo CV!")
    time.sleep(1) # Đợi 1 giây cho popup hiệu ứng mờ dần và biến mất hẳn
    
except NoSuchElementException:
    # Nếu mở trang lên mà KHÔNG CÓ popup này thì cứ đi tiếp bình thường
    pass

# 2. Lấy link job trong 1 page

In [11]:
# Mở trang ITviec (trang 1)
url = 'https://itviec.com/it-jobs/data-analyst?page=1'
driver.get(url)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn

In [12]:
# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [13]:
# Lấy mã nguồn HTML của trang web
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

In [ ]:
soup

In [13]:
# 1. Tìm container chứa danh sách công việc BẰNG "jobList" (Dùng find)
job_data = soup.find("div", attrs={"data-search--pagination-target": "jobList"})

In [ ]:
# Giả sử job_data là biến chứa cái khung danh sách job (đã lấy ở bước trước)

if job_data:
    # 1. Tìm TẤT CẢ các thẻ <h3> có thuộc tính 'data-search--job-selection-target' là 'jobTitle'
    job_tab = job_data.find_all('h3', attrs={'data-search--job-selection-target': 'jobTitle'})
    
    # Đếm số lượng job
    print(f"Found {len(job_tab)} job listings on page 1")
    
    # (Tùy chọn) In thử thẻ div của job đầu tiên để xem cấu trúc
    # print(job_previews[0])
else:
    print("Không tìm thấy khung chứa danh sách job.")

Found 20 job listings on page 1


In [ ]:
job_tab

[<h3 class="imt-3 text-break" data-controller="utm-tracking" data-search--job-selection-target="jobTitle" data-url="https://itviec.com/it-jobs/senior-data-analyst-scientist-kosatec-empowering-global-innovation-4238?lab_feature=preview_jd_page" href="null">
 Senior Data Analyst/Scientist
 </h3>,
 <h3 class="imt-3 text-break" data-controller="utm-tracking" data-search--job-selection-target="jobTitle" data-url="https://itviec.com/it-jobs/senior-data-analyst-nab-innovation-centre-vietnam-2240?lab_feature=preview_jd_page" href="null">
 Senior Data Analyst
 </h3>,
 <h3 class="imt-3 text-break" data-controller="utm-tracking" data-search--job-selection-target="jobTitle" data-url="https://itviec.com/it-jobs/data-analyst-sql-bi-tools-for-edtech-project-cong-ty-tnhh-teenup-4903?lab_feature=preview_jd_page" href="null">
 Data Analyst (SQL, BI tools) for Edtech project
 </h3>,
 <h3 class="imt-3 text-break" data-controller="utm-tracking" data-search--job-selection-target="jobTitle" data-url="https:/

In [ ]:
job_link = job_tab[0].get('data-url')
job_link

'https://itviec.com/it-jobs/senior-data-analyst-scientist-kosatec-empowering-global-innovation-4238?lab_feature=preview_jd_page'

In [17]:
# list job
# Giả sử job_data là biến chứa cái khung danh sách job (đã lấy ở bước trước)

if job_data:
    # 1. Tìm TẤT CẢ các thẻ <h3> có thuộc tính 'data-search--job-selection-target' là 'jobTitle'
    job_tab = job_data.find_all('h3', attrs={'data-search--job-selection-target': 'jobTitle'})
    
    # Đếm số lượng job
    print(f"Found {len(job_previews)} job listings on page 1")



    job_link = []
    
    # 2. Lặp qua từng thẻ <h3> vừa tìm được
    for tag in job_tab:
        # Lấy giá trị của thuộc tính 'data-url' (nó chứa sẵn nguyên cái link dài thòong)
        link = tag.get('data-url')
        
        if link:
            job_link.append(link)

    # 3. In kết quả ra xem thử
    for i, url in enumerate(job_link, 1):
        print(f"{i}. {url}")

else:
    print("Không tìm thấy khung chứa danh sách job.")

Found 20 job listings on page 1
1. https://itviec.com/it-jobs/senior-data-analyst-scientist-kosatec-empowering-global-innovation-4238?lab_feature=preview_jd_page
2. https://itviec.com/it-jobs/senior-data-analyst-nab-innovation-centre-vietnam-2240?lab_feature=preview_jd_page
3. https://itviec.com/it-jobs/data-analyst-sql-bi-tools-for-edtech-project-cong-ty-tnhh-teenup-4903?lab_feature=preview_jd_page
4. https://itviec.com/it-jobs/data-analyst-power-bi-developer-cong-ty-tnhh-socotec-viet-nam-5759?lab_feature=preview_jd_page
5. https://itviec.com/it-jobs/data-analyst-data-division-mb-bank-4114?lab_feature=preview_jd_page
6. https://itviec.com/it-jobs/business-customer-data-analyst-data-division-mb-bank-3611?lab_feature=preview_jd_page
7. https://itviec.com/it-jobs/data-analyst-sql-python-database-sports-marketing-software-services-co-ltd-4027?lab_feature=preview_jd_page
8. https://itviec.com/it-jobs/supply-chain-data-analyst-total-up-to-34-000-usd-year-crossian-2250?lab_feature=preview_jd

In [18]:
job_link

['https://itviec.com/it-jobs/senior-data-analyst-scientist-kosatec-empowering-global-innovation-4238?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/senior-data-analyst-nab-innovation-centre-vietnam-2240?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/data-analyst-sql-bi-tools-for-edtech-project-cong-ty-tnhh-teenup-4903?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/data-analyst-power-bi-developer-cong-ty-tnhh-socotec-viet-nam-5759?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/data-analyst-data-division-mb-bank-4114?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/business-customer-data-analyst-data-division-mb-bank-3611?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/data-analyst-sql-python-database-sports-marketing-software-services-co-ltd-4027?lab_feature=preview_jd_page',
 'https://itviec.com/it-jobs/supply-chain-data-analyst-total-up-to-34-000-usd-year-crossian-2250?lab_feature=preview_jd_page',
 'https://itviec.c

# 3. Con trỏ chuột sang trang kế tiếp

In [23]:
KEYWORD = "data-analyst"
MAX_PAGES = 5 # Đặt số trang tối đa bạn muốn cào
all_job_links = [] # List tổng chứa tất cả các link thu thập được từ mọi trang

print(f"\n{'='*50}")
print(f"BẮT ĐẦU CÀO DANH SÁCH LINK CHO TỪ KHÓA: {KEYWORD.upper()}")
print(f"{'='*50}")


BẮT ĐẦU CÀO DANH SÁCH LINK CHO TỪ KHÓA: DATA-ANALYST


In [24]:
# Vòng lặp chạy từ trang 1 đến MAX_PAGES
for page in range(1, MAX_PAGES + 1):
    # 1. Tạo URL cho trang hiện tại
    url = f'https://itviec.com/it-jobs/{KEYWORD}?page={page}'
    print(f"\n---> Đang truy cập Trang {page}: {url}")

    # 3. Lấy mã nguồn HTML đưa cho BeautifulSoup
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')    
    
    # 5. KIỂM TRA ĐIỀU KIỆN DỪNG (BREAK)    
    if not job_data:
        print(f"[THÔNG BÁO] Không tìm thấy khung danh sách job ở trang {page}. Dừng cào!")
        break # Thoát khỏi vòng lặp for hoàn toàn


---> Đang truy cập Trang 1: https://itviec.com/it-jobs/data-analyst?page=1

---> Đang truy cập Trang 2: https://itviec.com/it-jobs/data-analyst?page=2

---> Đang truy cập Trang 3: https://itviec.com/it-jobs/data-analyst?page=3

---> Đang truy cập Trang 4: https://itviec.com/it-jobs/data-analyst?page=4

---> Đang truy cập Trang 5: https://itviec.com/it-jobs/data-analyst?page=5


In [ ]:
# Vòng lặp chạy từ trang 1 đến MAX_PAGES
for page in range(1, MAX_PAGES + 1):
    # 1. Tạo URL cho trang hiện tại
    url = f'https://itviec.com/it-jobs/{KEYWORD}?page={page}'
    print(f"\n---> Đang truy cập Trang {page}: {url}")
    
    driver.get(url)
    time.sleep(3) # Đợi trang load cơ bản

    # 2. Cuộn xuống cuối trang để load toàn bộ job (Lazy load)
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # 3. Lấy mã nguồn HTML đưa cho BeautifulSoup
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # 4. Tìm container chứa danh sách công việc
    job_data = soup.find("div", attrs={"data-search--pagination-target": "jobList"})

    # 5. KIỂM TRA ĐIỀU KIỆN DỪNG (BREAK)
    if not job_data:
        print(f"[THÔNG BÁO] Không tìm thấy khung danh sách job ở trang {page}. Dừng cào!")
        break # Thoát khỏi vòng lặp for hoàn toàn

    # 6. Tìm TẤT CẢ các thẻ <h3> chứa link job
    job_tab = job_data.find_all('h3', attrs={'data-search--job-selection-target': 'jobTitle'})
    
    # Nếu khung jobList tồn tại nhưng bên trong không có job nào (trang trống)
    if len(job_tab) == 0:
        print(f"[THÔNG BÁO] Trang {page} không có job nào. Đã hết dữ liệu. Dừng cào!")
        break

    print(f"[*] Tìm thấy {len(job_tab)} job trên trang {page}")

    # 7. Lặp qua từng thẻ <h3> để lấy link và đưa vào list tổng
    page_links_count = 0
    for tag in job_tab:
        link = tag.get('data-url')
        if link:
            clean_href = link.split('?')[0]
            all_job_links.append(clean_href)
            page_links_count += 1
            
    print(f"[*] Đã lưu {page_links_count} link từ trang {page} vào danh sách tổng.")


# ==========================================
# (Tùy chọn) In thử 5 link đầu tiên ra xem
for i, url in enumerate(all_job_links, 1):
    print(f"{i}. {url}")


---> Đang truy cập Trang 1: https://itviec.com/it-jobs/data-analyst?page=1
[*] Tìm thấy 20 job trên trang 1
[*] Đã lưu 20 link từ trang 1 vào danh sách tổng.

---> Đang truy cập Trang 2: https://itviec.com/it-jobs/data-analyst?page=2
[*] Tìm thấy 4 job trên trang 2
[*] Đã lưu 4 link từ trang 2 vào danh sách tổng.

---> Đang truy cập Trang 3: https://itviec.com/it-jobs/data-analyst?page=3
[THÔNG BÁO] Trang 3 không có job nào. Đã hết dữ liệu. Dừng cào!
1. https://itviec.com/it-jobs/data-analyst-power-bi-mssql-python-s4bt-solutions-for-business-travel-0526?lab_feature=preview_jd_page
2. https://itviec.com/it-jobs/supply-chain-data-analyst-total-up-to-34-000-usd-year-crossian-2250?lab_feature=preview_jd_page
3. https://itviec.com/it-jobs/data-analyst-sql-bi-tools-for-edtech-project-cong-ty-tnhh-teenup-4903?lab_feature=preview_jd_page
4. https://itviec.com/it-jobs/senior-data-analyst-scientist-kosatec-empowering-global-innovation-4238?lab_feature=preview_jd_page
5. https://itviec.com/it-jo

# 4. Bắt đầu lấy data sản phẩm nè

### Nhóm 1: Các trường cào thành công từ web (Có dữ liệu thực tế)
1.	`job_url`: Link chi tiết của job đó.
2.	`job_title`: Tiêu đề công việc.
3.	`company_name`: Tên công ty.
4.	`salary`: Mức lương (Code có bước Login ở đầu để xem được lương, vì ITviec thường ẩn lương nếu không đăng nhập).

5.  `job_intern_fresher`: sẽ có dòng chữ "Fresher Accepted"
6.	`location`: Địa điểm làm việc.
7.  `job_type`: at office / Hybrid / Remote
8.	`date_post`: Thời gian up job (Timestamp). Phần deadline, sẽ làm trong phần ETL, dự đoán job đăng trong 1 tháng.

9.  `job_skills`: Data Analysis, PL/SQL, Oracle, Data modeling, ETL, Business Intelligence
10. `job_expertise`: vì có job ghi chuyên môn "`data engineer`" trong khi keyword search là "`data-analyst`"
11. `job_domain`: ví dụ 'Banking', 'E-commerce', 'IT Services and IT Consulting', 'Software Products and Web Services', 'AI Software & Services'

12.	`job_description`: Mô tả công việc.
13.	`requirements`: Yêu cầu công việc (Kinh nghiệm, kỹ năng).
14.	`benefits`: Phúc lợi (Lý do tại sao bạn sẽ yêu thích làm việc tại đây).

15.	`company_industry`: Lĩnh vực của công ty (ví dụ: Software Products and Web Services)
16. `company_type`: Trong ngành IT, sự khác biệt giữa Product (Sản phẩm) và Outsourcing (Gia công) là cực kỳ lớn. Nếu bạn có trường dữ liệu này, bạn có thể vẽ ra những biểu đồ trả lời các câu hỏi kinh doanh (Business Questions) cực kỳ thực tế
17. `company_country`: vietnam, japan ... để biết yêu cầu của cty nước ngoài khác vs cty thuần việt như thế nào



### Nhóm 2: Các trường bị bỏ qua (Gán mặc định là None)

Do đặc thù giao diện của ITviec không hiển thị rõ ràng, nên các trường sau, ko có cào, ko có xuất hiện trong code này

1.	experience_level (Cấp bậc kinh nghiệm)
2.	education_level (Trình độ học vấn)
3.	job_deadline (Hạn nộp hồ sơ)

In [12]:
# https://itviec.com/it-jobs/data-analyst-p-l-analytics-fe-credit-4306?lab_feature=preview_jd_page

job_link = 'https://itviec.com/it-jobs/data-analyst-p-l-analytics-fe-credit-4306?lab_feature=preview_jd_page'
job_link = job_link.split('?')[0]
job_link

'https://itviec.com/it-jobs/data-analyst-p-l-analytics-fe-credit-4306'

In [13]:

# Mở trang job chi tiết
driver.get(job_link)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn


# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [14]:
# Lấy mã nguồn HTML của trang web
new_html = driver.page_source
new_soup = BeautifulSoup(new_html, 'html.parser')

In [ ]:
new_soup

## 4.1	Title Job

In [14]:
# 1. Tìm khối chứa thông tin header (Mỏ neo bền vững)
title_tag_big = new_soup.find('div', class_='job-header-info')
title_tag_big

<div class="job-header-info">
<h1 class="ipt-xl-6 text-it-black">Data Analyst (P&amp;L Analytics)</h1>
<div class="employer-name">FE CREDIT</div>
<div class="d-flex align-items-center fw-700 ifs-18">
<div class="d-flex align-items-center salary text-success-color">
<svg fill="none" height="24" stroke="currentColor" viewbox="0 0 24 24" width="24" xmlns="http://www.w3.org/2000/svg">
<path d="M12.0044 6V18" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"></path>
<path d="M14.9287 7.90332H10.5395C9.99625 7.90332 9.47527 8.11912 9.09114 8.50325C8.70701 8.88738 8.49121 9.40836 8.49121 9.9516C8.49121 10.4948 8.70701 11.0158 9.09114 11.4C9.47527 11.7841 9.99625 11.9999 10.5395 11.9999H13.4656C14.0088 11.9999 14.5298 12.2157 14.914 12.5998C15.2981 12.9839 15.5139 13.5049 15.5139 14.0482C15.5139 14.5914 15.2981 15.1124 14.914 15.4965C14.5298 15.8806 14.0088 16.0964 13.4656 16.0964H8.49121" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"></pat

In [19]:
title_tag = title_tag_big.find('h1')
title_tag

<h1 class="ipt-xl-6 text-it-black">Data Analyst (P&amp;L Analytics)</h1>

In [20]:
title_tag.get_text(strip=True)

'Data Analyst (P&L Analytics)'

In [ ]:
# function 1: lấy tiêu đề job

def get_title(new_soup):
    try:
        title = new_soup.find('h1', attrs={'name': 'title'}).text.strip()
        
    except AttributeError:
        title = None

    return title

In [21]:
# function 1: lấy tiêu đề job

def get_title(soup):
    try:
        # 1. Tìm khối chứa thông tin header (Mỏ neo bền vững)
        header_info = soup.find('div', class_='job-header-info')
        
        if header_info:
            # 2. Tìm thẻ h1 bên trong khối đó
            title_tag = header_info.find('h1')
            
            if title_tag:
                # 3. Lấy text và xóa khoảng trắng thừa
                return title_tag.get_text(strip=True)
                
        return None
    except Exception as e:
        # Ghi log lỗi nếu cần
        logging.error(f"Lỗi khi lấy title: {e}")
        return None

In [22]:
# test function: lấy tiêu đề job
title = get_title(new_soup)
title

'Data Analyst (P&L Analytics)'

## 4.2	Company

In [15]:
# Lấy tên cty
company_title = title_tag_big.find('div', class_='employer-name').text.strip()
company_title

'FE CREDIT'

In [ ]:
# function 2: lấy tên cty

def get_company(new_soup):
    try:
        title_tag_big = new_soup.find('div', class_='job-header-info')
        company = title_tag_big.find('div', class_='employer-name').text.strip()
        
    except AttributeError:
        company = None

    return company

In [17]:
# test function: lấy tên cty
company = get_company(new_soup)
company

'FE CREDIT'

## 4.3  Salary

In [22]:
title_tag_big = new_soup.find('div', class_='job-header-info')

salary_level_tag = title_tag_big.find('div', class_='salary')
salary_level_tag

<div class="d-flex align-items-center salary text-success-color">
<svg fill="none" height="24" stroke="currentColor" viewbox="0 0 24 24" width="24" xmlns="http://www.w3.org/2000/svg">
<path d="M12.0044 6V18" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"></path>
<path d="M14.9287 7.90332H10.5395C9.99625 7.90332 9.47527 8.11912 9.09114 8.50325C8.70701 8.88738 8.49121 9.40836 8.49121 9.9516C8.49121 10.4948 8.70701 11.0158 9.09114 11.4C9.47527 11.7841 9.99625 11.9999 10.5395 11.9999H13.4656C14.0088 11.9999 14.5298 12.2157 14.914 12.5998C15.2981 12.9839 15.5139 13.5049 15.5139 14.0482C15.5139 14.5914 15.2981 15.1124 14.914 15.4965C14.5298 15.8806 14.0088 16.0964 13.4656 16.0964H8.49121" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"></path>
<circle cx="12.0002" cy="12.0002" r="9.3" stroke="currentColor" stroke-width="2"></circle>
</svg>
<span class="ips-2 fw-500">500 - 900 USD</span>
</div>

In [24]:
salary_level = salary_level_tag.find('span')
salary_level

<span class="ips-2 fw-500">500 - 900 USD</span>

In [26]:
salary = salary_level.get_text(strip=True)
salary

'500 - 900 USD'

In [27]:
# function 3: lấy mức lương

def get_salary(soup):
    try:
        # 1. Tìm thẻ div có chứa class 'salary' (Mỏ neo bền vững)
        salary_div = soup.find('div', class_='salary')
        
        if salary_div:
            # 2. Tìm thẻ span nằm bên trong div salary đó
            salary_span = salary_div.find('span')
            
            if salary_span:
                # 3. Lấy text và xóa khoảng trắng thừa
                return salary_span.get_text(strip=True)
                
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy lương: {e}")
        return None

In [28]:
salary = get_salary(new_soup)
salary

'500 - 900 USD'

## 4.4  job_intern_fresher

In [ ]:
# 1. Tìm khối cha chứa toàn bộ thông tin tóm tắt (Mỏ neo)
info_box = new_soup.find('div', class_='job-show-info')
info_box

In [31]:
fresher_tag = info_box.find('a', string=re.compile(r'Fresher Accepted', re.IGNORECASE))
fresher_tag

<a class="text-reset" data-controller="utm-tracking" href="/it-jobs/fresher-accepted?click_source=Skill+tag">Fresher Accepted</a>

In [ ]:
# function 4: Xác định job đó có phải job fresher / intern ko ?

def get_is_fresher(soup):
    try:
        # 1. Tìm khối cha chứa toàn bộ thông tin tóm tắt (Mỏ neo)
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Chỉ tìm chữ "Fresher Accepted" BÊN TRONG khối info_box này
            # Dùng re.IGNORECASE để không phân biệt hoa/thường
            fresher_tag = info_box.find('a', string=re.compile(r'Fresher Accepted', re.IGNORECASE))
            
            if fresher_tag:
                return "Yes"
                
        return "No"
    except Exception as e:
        logging.error(f"Lỗi khi check Fresher: {e}")
        return "No"

In [33]:
is_fresher = get_is_fresher(new_soup)
is_fresher

'Yes'

### 4.4.2 test vs case ko phải fresher

In [34]:
job_link = 'https://itviec.com/it-jobs/chuyen-vien-phan-tich-nghiep-vu-du-lieu-fetek-technology-company-limited-5050'

In [37]:
is_fresher_test = get_is_fresher(new_soup)
is_fresher_test

'No'

## 4.5  Address

In [43]:
# function 5: Lấy địa chỉ cty, sau sẽ ETL riêng để lấy tỉnh/ thành phố

def get_location(soup):
    try:
        # 1. Tìm khối thông tin chung (Mỏ neo lớn)
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Tìm thẻ <a> có chứa link Google Maps bên trong khối info_box
            map_link = info_box.find('a', href=lambda href: href and "google.com/maps" in href)
            
            if map_link:
                # 3. Lấy thẻ <div> cha chứa cả link map và địa chỉ
                parent_div = map_link.find_parent('div')
                
                # 4. Tìm thẻ <span> chứa địa chỉ bên trong thẻ div cha đó
                # (Thường nó là thẻ span duy nhất có class 'normal-text' trong div này)
                address_span = parent_div.find('span', class_='normal-text')
                
                if address_span:
                    return address_span.get_text(strip=True)
                    
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy địa chỉ: {e}")
        return None

In [41]:
location = get_location(new_soup)
location

'Số 3, ngõ 98 Vũ Trọng Phụng, Thanh Xuan, Ha Noi'

## 4.6  job_type
at office / Hybrid / Remote

In [42]:
# function 6: Lấy loại hình công việc dựa trên từ khóa có sẵn của ITviec

def get_job_type(soup):
    try:
        # 1. Vẫn giữ nguyên tắc: Giới hạn phạm vi tìm kiếm trong khối info_box
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Tạo một mẫu Regex chứa 3 từ khóa có thể xảy ra. 
            # re.IGNORECASE giúp bắt được cả "At Office", "at office", "REMOTE"...
            pattern = re.compile(r'(At office|Hybrid|Remote)', re.IGNORECASE)
            
            # 3. Tìm đoạn text nào khớp với 1 trong 3 từ khóa trên
            matched_text = info_box.find(string=pattern)
            
            if matched_text:
                # Trả về chính xác đoạn text hiển thị trên web (đã xóa khoảng trắng thừa)
                return matched_text.strip()
                
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy Job Type: {e}")
        return None

In [44]:
job_type = get_job_type(new_soup)
job_type

'At office'

### 4.6.2 Test vs các link khác 

In [45]:
link_1 = 'https://itviec.com/it-jobs/senior-data-architect-engineer-microsoft-fabric-huenx-vietnam-0415'
link_2 = 'https://itviec.com/it-jobs/data-analyst-power-bi-mssql-python-s4bt-solutions-for-business-travel-0526'

In [50]:
driver.get(link_1)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn

# Lấy mã nguồn HTML của trang web
new_html_test = driver.page_source
new_soup_test = BeautifulSoup(new_html_test, 'html.parser')

job_type_test = get_job_type(new_soup_test)
job_type_test

'Remote'

## 4.7  date_post
Là ngày hôm nay trừ đi số ngày đã đăng

In [19]:
# 1. Tìm khối thông tin chung (Mỏ neo)
info_box = new_soup.find('div', class_='job-show-info')

if info_box:
    # 2. Tìm đoạn text có chứa chữ "Posted"
    # Dùng re.compile để tìm kiếm linh hoạt (không phân biệt hoa/thường)
    posted_text_node = info_box.find(string=re.compile(r'Posted', re.IGNORECASE))
    
    if posted_text_node:
        # Lấy toàn bộ chuỗi text (ví dụ: "Posted 1 day ago" hoặc "Posted 3 days ago")
        raw_text = posted_text_node.strip()
        print(raw_text)

Posted
1 day ago


In [22]:
# 3. Dùng Regex để trích xuất con số ra khỏi chuỗi
# \d+ sẽ tìm một hoặc nhiều chữ số liên tiếp
match = re.search(r'\d+', raw_text)

if match:
    # Chuyển con số tìm được thành kiểu số nguyên (integer)
    number = int(match.group())
    now = datetime.datetime.now()
    print(number)

1


In [23]:
if 'hour' in raw_text:
    # Nếu hết hạn trong vài giờ, nghĩa là hạn chót vẫn là ngày hôm nay
    date_posted = now

elif 'day' in raw_text:
    # Cộng thêm số ngày tương ứng
    date_posted = now - datetime.timedelta(days=number)
else:
    # Các trường hợp khác không xác định
    date_posted = None


date_posted

datetime.datetime(2026, 4, 4, 0, 41, 16, 352665)

In [24]:
date_posted.strftime('%Y-%m-%d')

'2026-04-04'

In [ ]:
# function 7: lấy ngày post job 

def get_date_posted(soup):
    try:
        # 1. Tìm khối thông tin chung (Mỏ neo)
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Tìm đoạn text có chứa chữ "Posted"
            # Dùng re.compile để tìm kiếm linh hoạt (không phân biệt hoa/thường)
            posted_text_node = info_box.find(string=re.compile(r'Posted', re.IGNORECASE))
            
            if posted_text_node:
                # Lấy toàn bộ chuỗi text (ví dụ: "Posted 1 day ago" hoặc "Posted 3 days ago")
                raw_text = posted_text_node.strip()
                
                # 3. Dùng Regex để trích xuất con số ra khỏi chuỗi
                # \d+ sẽ tìm một hoặc nhiều chữ số liên tiếp
                match = re.search(r'\d+', raw_text)
                
                if match:
                    # Chuyển con số tìm được thành kiểu số nguyên (integer)
                    number = int(match.group())
                    
                    # 4. Tính toán ngày đăng: Lấy ngày hôm nay TRỪ ĐI số ngày đã đăng
                    now = datetime.datetime.now()
                    
                    
                    if 'hour' in raw_text:
                        # Nếu hết hạn trong vài giờ, nghĩa là hạn chót vẫn là ngày hôm nay
                        date_posted = now

                    elif 'day' in raw_text:
                        # Cộng thêm số ngày tương ứng
                        date_posted = now - datetime.timedelta(days=number)
                    else:
                        # Các trường hợp khác không xác định
                        return None
                        
                    
                    # 5. Trả về định dạng chuẩn YYYY-MM-DD (ví dụ: 2024-05-15)
                    return date_posted.strftime('%Y-%m-%d')

    
    except Exception as e:
        logging.error(f"Lỗi khi tính ngày đăng: {e}")
        return None

In [26]:
date_posted = get_date_posted(new_soup)
date_posted

'2026-04-04'

### 4.7.2 test case khác

In [ ]:
# Test các job khác

link_1 = 'https://itviec.com/it-jobs/senior-data-architect-engineer-microsoft-fabric-huenx-vietnam-0415'
link_2 = 'https://itviec.com/it-jobs/business-customer-data-analyst-data-division-mb-bank-3611'

driver.get(link_2)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn

# Lấy mã nguồn HTML của trang web
new_html_test = driver.page_source
new_soup_test = BeautifulSoup(new_html_test, 'html.parser')

date_posted_test = get_date_posted(new_soup_test)
date_posted_test

## 4.8  job_skills

In [ ]:
# function 8: Lấy các skills cần có: English, SQL, Python, ... 

def get_skills(soup):
    try:
        # 1. Giới hạn phạm vi tìm kiếm trong khối info_box
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Tìm thẻ div có chứa chính xác chữ "Skills:"
            # Dùng re.compile để tìm kiếm linh hoạt, bỏ qua khoảng trắng thừa
            skills_label = info_box.find('div', string=re.compile(r'Skills:', re.IGNORECASE))
            
            if skills_label:
                # 3. Tìm thẻ div cha bao bọc cả chữ "Skills:" và danh sách các thẻ <a>
                parent_div = skills_label.find_parent('div')
                
                if parent_div:
                    # 4. Tìm tất cả các thẻ <a> bên trong thẻ div cha này
                    skill_tags = parent_div.find_all('a')
                    
                    # 5. Lấy text từ các thẻ <a>, xóa khoảng trắng và gom vào một list
                    skills_list = [tag.get_text(strip=True) for tag in skill_tags]
                    
                    # 6. Nối các kỹ năng lại thành một chuỗi, cách nhau bằng dấu phẩy
                    # Ví dụ: "SQL, English"
                    if skills_list:
                        return ", ".join(skills_list)
                        
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy Skills: {e}")
        return None

In [16]:
skill = get_skills(new_soup)
skill

'SQL, English'

In [18]:
# Test các job khác

link_1 = 'https://itviec.com/it-jobs/senior-data-architect-engineer-microsoft-fabric-huenx-vietnam-0415'
link_2 = 'https://itviec.com/it-jobs/business-customer-data-analyst-data-division-mb-bank-3611'

driver.get(link_1)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn

# Lấy mã nguồn HTML của trang web
new_html_test = driver.page_source
new_soup_test = BeautifulSoup(new_html_test, 'html.parser')

skill_test = get_skills(new_soup_test)
skill_test

'ETL, ReactJS, Data modeling, Data Warehousing, C#, Python'

## 4.9  job_expertise

In [ ]:
# function 9: Lấy chuyên môn của job 

def get_job_expertise(soup):
    try:
        # 1. Giới hạn phạm vi tìm kiếm trong khối info_box (Mỏ neo lớn)
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Tìm thẻ div có chứa chính xác chữ "Job Expertise:" (Mỏ neo nhỏ)
            # Dùng re.compile để tìm kiếm linh hoạt, không phân biệt hoa/thường
            expertise_label = info_box.find('div', string=re.compile(r'Job Expertise:', re.IGNORECASE))
            
            if expertise_label:
                # 3. Tìm thẻ div cha bao bọc cả chữ "Job Expertise:" và các thẻ <a> chứa giá trị
                parent_div = expertise_label.find_parent('div')
                
                if parent_div:
                    # 4. Tìm tất cả các thẻ <a> bên trong thẻ div cha này
                    expertise_tags = parent_div.find_all('a')
                    
                    # 5. Lấy text từ các thẻ <a>, xóa khoảng trắng và gom vào một list
                    expertise_list = [tag.get_text(strip=True) for tag in expertise_tags]
                    
                    # 6. Nối các chuyên môn lại thành một chuỗi, cách nhau bằng dấu phẩy
                    # Ví dụ: "Data Analyst, Business Analyst"
                    if expertise_list:
                        return ", ".join(expertise_list)
                        
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy Job Expertise: {e}")
        return None

In [21]:
expertise = get_job_expertise(new_soup)
expertise

'Data Analyst'

## 4.10 job_domain

In [22]:
# function 10: Lấy domain của job 

def get_job_domain(soup):
    try:
        # 1. Giới hạn phạm vi tìm kiếm trong khối info_box (Mỏ neo lớn)
        info_box = soup.find('div', class_='job-show-info')
        
        if info_box:
            # 2. Tìm thẻ div có chứa chính xác chữ "Job Domain:" (Mỏ neo nhỏ)
            domain_label = info_box.find('div', string=re.compile(r'Job Domain:', re.IGNORECASE))
            
            if domain_label:
                # 3. Tìm thẻ div cha bao bọc cả chữ "Job Domain:" và các thẻ chứa giá trị
                parent_div = domain_label.find_parent('div')
                
                if parent_div:
                    # 4. THAY ĐỔI QUAN TRỌNG: Tìm các thẻ <div> có class 'itag'
                    # Đây là các thẻ chứa tên Domain (Banking, Financial Services...)
                    domain_tags = parent_div.find_all('div', class_='itag')
                    
                    domain_list = []
                    for tag in domain_tags:
                        # 5. Lấy text trực tiếp của thẻ div đó, bỏ qua các thẻ con (như svg icon)
                        # Dùng tham số recursive=False để chỉ lấy text ở tầng ngoài cùng
                        # Ví dụ: Lấy "Financial Services" mà không lấy text ẩn trong icon "i"
                        text = tag.find(string=True, recursive=False)
                        
                        if text:
                            clean_text = text.strip()
                            if clean_text:
                                domain_list.append(clean_text)
                    
                    # 6. Nối các domain lại thành một chuỗi, cách nhau bằng dấu phẩy
                    if domain_list:
                        return ", ".join(domain_list)
                        
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy Job Domain: {e}")
        return None

In [23]:
job_domain = get_job_domain(new_soup)
job_domain

'Banking, Financial Services'

In [25]:
# Test các job khác

link_1 = 'https://itviec.com/it-jobs/senior-data-architect-engineer-microsoft-fabric-huenx-vietnam-0415'
link_2 = 'https://itviec.com/it-jobs/business-customer-data-analyst-data-division-mb-bank-3611'

driver.get(link_1)
time.sleep(3) # Đợi 3 giây để trang web tải hoàn toàn

# Lấy mã nguồn HTML của trang web
new_html_test = driver.page_source
new_soup_test = BeautifulSoup(new_html_test, 'html.parser')

job_domain_test = get_job_domain(new_soup_test)
job_domain_test

'IT Services and IT Consulting'

## 4.11 company_industry
lấy ở bên thông tin của cty

## 4.12 company_type
Trong ngành IT, sự khác biệt giữa Product (Sản phẩm) và Outsourcing (Gia công) là cực kỳ lớn. Nếu bạn có trường dữ liệu này, bạn có thể vẽ ra những biểu đồ trả lời các câu hỏi kinh doanh (Business Questions) cực kỳ thực tế

- `So sánh Mức lương (Salary)`: Làm Data Analyst cho công ty Product lương có cao hơn công ty Outsourcing không? (Thường công ty Product có dải lương base cao hơn, trong khi Outsourcing phụ thuộc vào dự án).
So sánh Kỹ năng (Skillset):

- `Công ty Product` thường yêu cầu DA tập trung sâu vào: A/B Testing, Product Metrics (LTV, CAC, Retention), User Behavior, Product Sense.

- `Công ty Outsourcing` thường yêu cầu DA: Đa năng, biết nhiều tool (PowerBI, Tableau, SQL đa nền tảng), giao tiếp tiếng Anh tốt để làm việc với khách hàng nước ngoài.

- `Xu hướng thị trường`: Hiện tại thị trường tuyển dụng DA ở Việt Nam đang nghiêng về Product hay Outsourcing?

## 4.13 company_country

In [26]:
# Cào cùng lúc 3 mục, vì ở trong cùng 1 box

def get_employer_info(soup, label_name):
    """
    Hàm đa năng để lấy thông tin công ty dựa trên tên nhãn (label).
    Ví dụ: label_name = "Company type"
    """
    try:
        # 1. Tìm khối chứa thông tin công ty (Mỏ neo lớn)
        employer_box = soup.find('section', class_='job-show-employer-info')
        
        if employer_box:
            # 2. Tìm thẻ div chứa cái nhãn (Label) cần tìm
            # Dùng re.compile để tìm kiếm linh hoạt, không phân biệt hoa/thường
            label_div = employer_box.find('div', string=re.compile(label_name, re.IGNORECASE))
            
            if label_div:
                # 3. Tìm thẻ div cha (thẻ <div class="row">) bao bọc cả Label và Value
                row_div = label_div.find_parent('div', class_='row')
                
                if row_div:
                    # 4. Tìm thẻ div chứa giá trị (Value)
                    # Thường nó là thẻ div thứ 2 trong row, có class chứa 'text-end'
                    value_div = row_div.find('div', class_=lambda c: c and 'text-end' in c)
                    
                    if value_div:
                        # 5. Lấy text và làm sạch
                        # Dùng separator=' ' để tránh dính chữ nếu có nhiều thẻ con bên trong
                        raw_text = value_div.get_text(separator=' ', strip=True)
                        
                        # Làm sạch khoảng trắng thừa
                        clean_text = ' '.join(raw_text.split())
                        return clean_text
                        
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy {label_name}: {e}")
        return None

# --- CÁCH SỬ DỤNG TRONG VÒNG LẶP CHÍNH ---
# Khi bạn đang cào chi tiết 1 job và đã có đối tượng `soup`:

# Lấy Company Type
company_type = get_employer_info(new_soup, "Company type")

# Lấy Company Industry
company_industry = get_employer_info(new_soup, "Company industry")

# Lấy Country
company_country = get_employer_info(new_soup, "Country")

In [28]:
company_type

'IT Product'

In [29]:
company_industry

'Financial Services'

In [30]:
company_country

'Vietnam'

## 4.13 MÔ TẢ CÔNG VIỆC

## 4.14 YÊU CẦU

## 4.15 PHÚC LỢI

In [ ]:
# Cào cùng lúc 3 mục, vì ở trong cùng 1 box

def extract_paragraph_content(soup, heading_keyword):
    """
    Hàm đa năng: Tìm thẻ h2 chứa keyword, sau đó lấy toàn bộ text 
    của các thẻ anh em (siblings) nằm dưới nó trong cùng 1 div.
    """
    try:
        # 1. Tìm thẻ h2 chứa từ khóa (Mỏ neo) - Không phân biệt hoa thường
        h2_tag = soup.find('h2', string=re.compile(heading_keyword, re.IGNORECASE))
        
        if h2_tag:
            content_parts = []
            
            # 2. Lấy tất cả các thẻ nằm SAU thẻ h2 này (p, ul, li...)
            for sibling in h2_tag.find_next_siblings():
                # Lấy text, dùng dấu cách để các chữ không bị dính vào nhau
                text = sibling.get_text(separator=' ', strip=True)
                if text:
                    content_parts.append(text)
            
            # 3. Nối tất cả lại thành 1 đoạn văn duy nhất
            raw_text = ' '.join(content_parts)
            
            # 4. Làm sạch dữ liệu (Data Cleansing)
            # Xóa các ký tự bullet point, gạch ngang đầu dòng
            clean_text = re.sub(r'[•\-\*·]', '', raw_text)
            # Xóa ký tự khoảng trắng ẩn (&nbsp; hay \xa0 trong HTML)
            clean_text = clean_text.replace('\xa0', ' ')
            # Chuẩn hóa khoảng trắng (biến nhiều dấu cách thành 1 dấu cách)
            final_text = ' '.join(clean_text.split())
            
            # Chống lỗi công thức Excel nếu text bắt đầu bằng dấu =, +, -, @
            if final_text.startswith(('-', '=', '+', '@')):
                final_text = "'" + final_text
                
            return final_text
            
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy {heading_keyword}: {e}")
        return None

# ==========================================
# ÁP DỤNG VÀO 3 HÀM CỦA BẠN
# ==========================================

def get_job_description(soup):
    # ITviec dùng chữ "Job description"
    return extract_paragraph_content(soup, "Job description")

def get_job_requirements(soup):
    # ITviec dùng chữ "Your skills and experience"
    return extract_paragraph_content(soup, "Your skills and experience")

def get_benefits(soup):
    # ITviec dùng chữ "Why you'll love working here"
    return extract_paragraph_content(soup, "Why you'll love working here")

In [32]:
job_description  = get_job_description(new_soup)
job_requirements = get_job_requirements(new_soup)
benefits         = get_benefits(new_soup)

In [33]:
job_description

'Coordinate with crossfunctional team to, build profit targets for products. Implement monitoring of results achieved compared to target. Build periodic reports related to product profitability, make recommendations to improve P&L Build revenue and expenses for pilot of products to be deployed. Evaluate product feasibility based on profitability analysis. Coordinate with departments to determine cost allocation methods, manage costs related to products. Undertake other tasks assigned by XSTU Analytics & Strategy'

In [34]:
job_requirements

'Bachelor degree or equivalent in Economics, Finance, Mathematics, Statistics or Computer Science. Degree from institution recognized internationally or top tier in the country of origin is preferred. For New Graduates: A Bachelor’s degree in a technical field (Computer Science, Engineering, Information Technology, or related disciplines). For Experienced Candidates: Proven professional experience in Portfolio Management, Finance, or Data Analysis. Knowledge of consumer finance products is a plus Proficiency in MS Office, SQL Understanding of Profit & Loss drivers and financial model Full English proficiency'

In [35]:
benefits

"See yourselves in a new light In essence of joining our company, you will be given the scope to seize every opportunity and helped to acquire specific competencies you’ll need to succeed either you wish to go wide or go deep. Our company prides ourselves on our underpinned principle of nurturing people in not only HR policies but also in our culture. We value innovations and we need people to work on initiatives and carry on our business vision of operating excellence and market leadership. Alongside with these, we believe our staff members deserve a good working condition, so that in addition to social health insurance you will be accessible to health and accident insurance and will be eligible to join in team building every year. We also value your efforts; for this reason, we give you both financial and nonfinancial rewards such as: KPI bonus, Best Initiatives, Best performer or Best manager of the year, etc. Company's Benefits Financial support: Probation with full salary Lunch al